# Stooq vs Yahoo — Divergence Visualization

Visual proof that Stooq's adjusted close drifts from Yahoo's `auto_adjust=True` close on high-yield names.

**Expected pattern:**
- Recent bars: Stooq ≈ Yahoo (small or no gap)
- Older bars: Stooq increasingly above Yahoo — missing post-snapshot dividend back-adjustment
- Ratio plot: drifts upward going back in time. Steps/jumps mark missed ex-dividend dates

Low-yield control (AAPL) should overlay almost perfectly.

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from plotly.subplots import make_subplots

from irp.core.config import config
from irp.data.stooq import prices

TICKERS = ['AVB', 'ARE', 'O', 'SPG', 'SO', 'T', 'AAPL']
START_DATE = '2005-01-01'

In [2]:
def load_stooq(ticker: str, start: str = START_DATE) -> pd.DataFrame:
    df = prices(tickers=ticker, start=start)
    if df.empty:
        return df
    df['Date'] = pd.to_datetime(df['Date'].astype(str), format='%Y%m%d')
    return df[['Date', 'C']].rename(columns={'C': 'Stooq'}).set_index('Date')


def load_yahoo(ticker: str, start: str = START_DATE) -> pd.DataFrame:
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False, threads=False)
    if df is None or df.empty:
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[['Close']].rename(columns={'Close': 'Yahoo'})


def compare(ticker: str, start: str = START_DATE) -> pd.DataFrame:
    s = load_stooq(ticker, start)
    y = load_yahoo(ticker, start)
    if s.empty or y.empty:
        return pd.DataFrame()
    df = s.join(y, how='inner')
    df['Ratio'] = df['Stooq'] / df['Yahoo']
    return df

In [3]:
def plot_compare(ticker: str, df: pd.DataFrame) -> go.Figure:
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.05,
        subplot_titles=(
            f'{ticker} — Close (log scale): Stooq vs Yahoo auto_adjust',
            f'{ticker} — Ratio Stooq / Yahoo',
        ),
    )
    fig.add_trace(
        go.Scatter(x=df.index, y=df['Stooq'], name='Stooq', line={'color': '#1f77b4', 'width': 1.2}),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df.index, y=df['Yahoo'], name='Yahoo adj', line={'color': '#ff7f0e', 'width': 1.2}),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df.index, y=df['Ratio'], name='Ratio', line={'color': '#d62728', 'width': 1.0}, showlegend=False),
        row=2, col=1,
    )
    fig.add_hline(y=1.0, line_dash='dash', line_color='gray', row=2, col=1)
    fig.update_yaxes(type='log', row=1, col=1)
    fig.update_yaxes(title_text='Ratio', row=2, col=1)
    fig.update_layout(
        height=520, width=1000,
        margin={'l': 50, 'r': 30, 't': 60, 'b': 30},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.04, 'xanchor': 'right', 'x': 1},
    )
    return fig

## Per-ticker comparison

Fetches Yahoo full history once per ticker (slow — budget ~5–10s per ticker). High-yield names at top, low-yield control (AAPL) at bottom.

In [4]:
import time

frames: dict[str, pd.DataFrame] = {}
for t in TICKERS:
    df = compare(t)
    if df.empty:
        print(f'{t}: no data')
        continue
    frames[t] = df
    last = df.iloc[-1]
    first = df.iloc[0]
    print(f'{t:6s}  bars={len(df):5d}  ratio_first={first["Ratio"]:.3f}  ratio_last={last["Ratio"]:.3f}  max={df["Ratio"].max():.3f}')
    time.sleep(1.5)

AVB     bars= 5338  ratio_first=1.424  ratio_last=1.000  max=1.424
ARE     bars= 5338  ratio_first=1.457  ratio_last=1.000  max=1.457
O       bars= 5338  ratio_first=1.819  ratio_last=1.000  max=1.820
SPG     bars= 5376  ratio_first=1.604  ratio_last=1.000  max=1.604
SO      bars= 5376  ratio_first=1.512  ratio_last=1.000  max=1.512
T       bars= 6380  ratio_first=1.653  ratio_last=0.000  max=1.653
AAPL    bars= 5376  ratio_first=1.001  ratio_last=1.000  max=1.011


In [5]:
for t, df in frames.items():
    plot_compare(t, df).show()

## Summary table

Snapshot of the divergence: ratio at start of series, at −5y / −3y / −1y, and most recent bar.

In [6]:
rows = []
today = pd.Timestamp.today().normalize()
anchors = {'first': None, '-15y': today - pd.DateOffset(years=15), '-5y': today - pd.DateOffset(years=5), '-1y': today - pd.DateOffset(years=1), 'last': None}
for t, df in frames.items():
    row = {'Ticker': t}
    for label, anchor in anchors.items():
        if label == 'first':
            row[label] = df['Ratio'].iloc[0]
        elif label == 'last':
            row[label] = df['Ratio'].iloc[-1]
        else:
            sub = df[df.index <= anchor]
            row[label] = sub['Ratio'].iloc[-1] if len(sub) else float('nan')
    rows.append(row)
summary = pd.DataFrame(rows).set_index('Ticker')
summary.style.format('{:.3f}').background_gradient(cmap='Reds', vmin=1.0, vmax=1.3)

,first,-15y,-5y,-1y,last
Ticker,,,,,
AVB,1.424,1.136,1.135,1.039,1.000
ARE,1.457,1.192,1.192,1.066,1.000
O,1.819,1.252,1.248,1.056,1.000
SPG,1.604,1.264,1.189,1.050,1.000
SO,1.512,1.124,1.124,1.033,1.000
T,1.653,1.480,1.233,0.001,0.000
AAPL,1.001,1.001,1.001,1.001,1.000
